STEP 1 — Download Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nltkdata/timitcorpus")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'timitcorpus' dataset.
Path to dataset files: /kaggle/input/timitcorpus


STEP 2 — Explore Files

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print("Folder:", root)
    print("Sample files:", files[:5])
    break

Folder: /kaggle/input/timitcorpus
Sample files: ['README']


STEP 3 — Get Audio + PHN Files

In [ ]:
import os

wav_files = []
phn_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        file_lower = file.lower()

        if file_lower.endswith(".wav"):
            wav_files.append(os.path.join(root, file))

        if file_lower.endswith(".phn"):
            phn_files.append(os.path.join(root, file))

print("Total WAV files:", len(wav_files))
print("Total PHN files:", len(phn_files))

print("WAV sample:", wav_files[:3])
print("PHN sample:", phn_files[:3])

Total WAV files: 160
Total PHN files: 160
WAV sample: ['/kaggle/input/timitcorpus/timit/timit/dr7-madd0/si1295.wav', '/kaggle/input/timitcorpus/timit/timit/dr7-madd0/si538.wav', '/kaggle/input/timitcorpus/timit/timit/dr7-madd0/sa1.wav']
PHN sample: ['/kaggle/input/timitcorpus/timit/timit/dr7-madd0/sx448.phn', '/kaggle/input/timitcorpus/timit/timit/dr7-madd0/si1798.phn', '/kaggle/input/timitcorpus/timit/timit/dr7-madd0/sx358.phn']


STEP 4 — Load One Audio File

In [ ]:
import librosa

audio_path = wav_files[0]
signal, sr = librosa.load(audio_path, sr=None)

print("Signal shape:", signal.shape)
print("Sample rate:", sr)

Signal shape: (29594,)
Sample rate: 16000


STEP 5 — Read PHN File

In [ ]:
phn_path = phn_files[0]

phonemes = []
with open(phn_path, "r") as f:
    for line in f:
        start, end, ph = line.strip().split()
        phonemes.append(ph)

print("Phonemes:", phonemes[:10])

Phonemes: ['h#', 's', 'f', 'ih', 'r', 'ix', 'kcl', 'k', 'el', 'gcl']


STEP 6 — Extract MFCC Features, Prepare data

In [ ]:
import librosa
import numpy as np

phn_path = phn_files[0]
audio_path = wav_files[0]

signal, sr = librosa.load(audio_path, sr=None)

segments = []
with open(phn_path, "r") as f:
    for line in f:
        start, end, ph = line.strip().split()
        start_sample = int(start)
        end_sample = int(end)
        segment = signal[start_sample:end_sample]
        if len(segment) > 0:
            n_fft = min(2048, len(segment))
            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13, n_fft=n_fft)
            segments.append(mfcc.T)

X = np.vstack(segments)
print("X shape after phoneme alignment:", X.shape)

X shape after phoneme alignment: (72, 13)


**STEP** 7 — Train HMM

In [ ]:
!pip install hmmlearn
from hmmlearn import hmm

model = hmm.GaussianHMM(n_components=5, covariance_type="diag", n_iter=100)

model.fit(X)

print("Model trained")

Model trained


STEP 8 — Inspect Model

In [ ]:
print("Transition Matrix:\n", model.transmat_)
print("\nInitial Probabilities:\n", model.startprob_)
print("\nMeans:\n", model.means_)
print("\nCovariances (Emission Matrix):\n", model.covars_)

Transition Matrix:
 [[8.34545191e-001 3.32913960e-002 3.32866940e-002 6.65687492e-002
  3.23079696e-002]
 [8.24622745e-002 8.61199960e-001 1.24437835e-092 2.77630107e-002
  2.85747544e-002]
 [1.13781009e-046 2.16419487e-108 8.74999611e-001 1.25000389e-001
  2.21290027e-318]
 [4.76697682e-002 1.43034748e-001 2.65053977e-042 8.09295484e-001
  9.10671335e-033]
 [1.25663716e-001 1.25663048e-001 2.40700200e-210 1.28176910e-035
  7.48673236e-001]]

Initial Probabilities:
 [3.96460740e-209 7.89693028e-254 1.00000000e+000 1.13345964e-105
 0.00000000e+000]

Means:
 [[-3.43243747e+02  1.55103894e+02 -6.15875048e+01 -3.09448992e+01
  -3.61435929e+01 -2.43144094e+01 -1.16804535e+01 -3.21524927e+01
  -1.06449417e+01 -9.81956236e+00  7.23680434e-01  1.77820474e-02
  -6.94618756e+00]
 [-3.90641229e+02  7.56869816e+01 -4.67160779e+01  6.21505449e+01
  -2.60313590e+01 -2.96070102e+01 -2.02226396e+01 -2.48690421e+01
  -5.43791603e+00 -3.35021189e+01 -8.89966131e+00 -4.69917773e+00
  -1.72165175e+01]
 [-

**STEP** 9 — Implement Forward Algorithm

In [35]:
import numpy as np

def log_gaussian_prob(x, mean, cov):
    cov = np.clip(cov, 1e-6, None)
    diff = x - mean
    return -0.5 * np.sum((diff ** 2) / cov) - 0.5 * np.sum(np.log(2 * np.pi * cov))

def forward_algorithm(obs, start_prob, trans_prob, means, covars):
    T = len(obs)
    N = len(start_prob)
    log_alpha = np.zeros((T, N))

    for i in range(N):
        log_alpha[0, i] = np.log(start_prob[i] + 1e-10) + log_gaussian_prob(obs[0], means[i], covars[i])

    for t in range(1, T):
        for j in range(N):
            log_sum = np.logaddexp.reduce(
                log_alpha[t-1] + np.log(trans_prob[:, j] + 1e-10)
            )
            log_alpha[t, j] = log_gaussian_prob(obs[t], means[j], covars[j]) + log_sum

    return np.logaddexp.reduce(log_alpha[T-1])

STEP 10 — Apply Forward Algorithm

In [36]:
likelihood = forward_algorithm(
    X,
    model.startprob_,
    model.transmat_,
    model.means_,
    model.covars_
)
print("Log Likelihood:", likelihood)

Log Likelihood: -4165594479725.575


STEP 11 — Train Multiple Word Models

In [37]:
models = {}

for i in range(3):
    signal, sr = librosa.load(wav_files[i], sr=None)
    phn_path = phn_files[i]
    segments = []
    with open(phn_path, "r") as f:
        for line in f:
            start, end, ph = line.strip().split()
            segment = signal[int(start):int(end)]
            if len(segment) > 0:
                n_fft = min(2048, len(segment))
                mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13, n_fft=n_fft)
                segments.append(mfcc.T)
    X = np.vstack(segments)

    model = hmm.GaussianHMM(n_components=5, covariance_type="diag", n_iter=50)
    model.fit(X)
    models[wav_files[i]] = model

print("Models trained:", len(models))

/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


Models trained: 3


/usr/local/lib/python3.12/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


STEP 12 — Predict Word Using Forward

In [39]:
test_signal, sr = librosa.load(wav_files[3], sr=None)
mfcc_test = librosa.feature.mfcc(y=test_signal, sr=sr, n_mfcc=13)
X_test = mfcc_test.T

scores = {}

for name, model in models.items():
    score = forward_algorithm(
        X_test,
        model.startprob_,
        model.transmat_,
        model.means_,
        model.covars_ # Added the missing 'covars' argument
    )
    scores[name] = score

predicted = max(scores, key=scores.get)

print("Prediction:", predicted)

Prediction: /kaggle/input/timitcorpus/timit/timit/dr7-madd0/si1295.wav
